# 第 2 课：TD 学习与 Q-learning

**学习目标**：用样本替代已知转移矩阵，构建自定义环境，写出完整的训练/评估循环。

价值迭代需要显式的 $P(s'|s,a)$。无模型 RL 则使用采样得到的四元组 $(s_t,a_t,r_t,s_{t+1})$。

Q-learning 的 TD 目标是

$$y_t=r_t+\gamma(1-\text{terminated}_t)\max_{a'}Q(s_{t+1},a').$$

更新：

$$Q(s_t,a_t)\leftarrow Q(s_t,a_t)+\alpha\big[y_t-Q(s_t,a_t)\big].$$

注意：`truncated=True`（如达到时间上限）不等于 MDP 真正终止；它通常**仍需要**对下一状态价值 bootstrap。

## 2.1 一个支持 reset / step 的最小环境

仿照 Gymnasium 的返回方式：`reset -> (obs, info)`，`step -> (obs, reward, terminated, truncated, info)`。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class LineWorld:
    def __init__(self, n=7, max_steps=25, slip=0.08):
        self.n, self.max_steps, self.slip = n, max_steps, slip
        self.rng = np.random.default_rng(0)

    def reset(self, seed=None):
        if seed is not None: self.rng = np.random.default_rng(seed)
        self.s, self.t = 0, 0
        return self.s, {}

    def step(self, action):
        assert action in (0, 1)
        if self.rng.random() < self.slip: action = 1 - action
        self.s = int(np.clip(self.s + (1 if action else -1), 0, self.n-1))
        self.t += 1
        terminated = (self.s == self.n - 1)
        truncated = (self.t >= self.max_steps) and not terminated
        reward = 1.0 if terminated else -0.01
        return self.s, reward, terminated, truncated, {}

env = LineWorld()
print(env.reset(seed=42))
print(env.step(1))

## 2.2 ε-greedy：探索与利用

训练早期随机探索，逐渐增大利用已学 Q 值的概率。每一轮 reset 代表新的 episode。

In [ ]:
rng = np.random.default_rng(2026)
Q = np.zeros((env.n, 2))
alpha, gamma = 0.15, 0.97
history = []
for ep in range(1800):
    s, _ = env.reset(seed=ep)
    epsilon = max(0.03, 0.6 * (1 - ep / 1800))
    total = 0.0
    while True:
        a = int(rng.integers(2)) if rng.random() < epsilon else int(np.argmax(Q[s]))
        ns, r, terminated, truncated, _ = env.step(a)
        target = r + gamma * (0.0 if terminated else np.max(Q[ns]))
        Q[s, a] += alpha * (target - Q[s, a])
        s, total = ns, total + r
        if terminated or truncated: break
    history.append(total)
print("Q:", np.round(Q, 2))
print("greedy action:", np.argmax(Q, axis=1))
plt.figure(figsize=(7,3))
plt.plot(np.convolve(history, np.ones(100)/100, mode="valid"))
plt.xlabel("Episode (moving average window 100)")
plt.ylabel("Return")
plt.show()

## 2.3 独立评估，而不是用训练回报当结论

In [ ]:
def evaluate(use_policy, seeds=range(10000, 10200)):
    returns = []
    test_env = LineWorld()
    random = np.random.default_rng(999)
    for seed in seeds:
        s, _ = test_env.reset(seed=seed)
        total = 0.0
        while True:
            a = int(np.argmax(Q[s])) if use_policy else int(random.integers(2))
            s, r, term, trunc, _ = test_env.step(a)
            total += r
            if term or trunc: break
        returns.append(total)
    return np.mean(returns)

trained_score, random_score = evaluate(True), evaluate(False)
print(f"Greedy: {trained_score:.3f}  Random: {random_score:.3f}")
assert trained_score > random_score

### 练习

1. 用一个 `replay_buffer` 列表存储每条 transition，理解 off-policy 的基本数据结构。
2. 禁止从时间上限 bootstrap，比较两种训练结果；解释为什么理论上这不是同一个值函数定义。
3. 从离散动作延伸至连续吹吸流量 $a\in[-1,1]$ 时，Q-table 为何不再合适？